## Install PySpark and create a session

In [118]:
!pip install pyspark==4.1.2

In [119]:
from pyspark.sql import SparkSession
from pyspark.sql import DataFrameReader

#create a spark session
spark=SparkSession.builder.appName("AirQuality").getOrCreate()
spark

## Read in data to pyspark dataframe

In [120]:
#Mount Drive
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [121]:
#Read in previously saved Parquet File
df_aq=spark.read.parquet("/content/drive/MyDrive/AirQualityModel/processed.pq")
df_aq.show(5)

+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+
|            SiteName|      Date|        SiteType|Hour|ParticulateReading|         WindSpeed|     WindDirection|Month|DayOfWeek|
+--------------------+----------+----------------+----+------------------+------------------+------------------+-----+---------+
|Peterborough Gart...|2025-01-01|Urban Background|   1|            12.618|13.699999809265137|             236.5|  Jan|      Wed|
|Peterborough Gart...|2025-01-01|Urban Background|   2|             3.325|14.199999809265137|             236.0|  Jan|      Wed|
|Peterborough Gart...|2025-01-01|Urban Background|   3|             2.453|14.199999809265137|235.89999389648438|  Jan|      Wed|
|Peterborough Gart...|2025-01-01|Urban Background|   4|             1.934|12.699999809265137|235.89999389648438|  Jan|      Wed|
|Peterborough Gart...|2025-01-01|Urban Background|   5|             1.722|12.600000381469727|234.

#Pre-Processing for pipeline

identify categorical, numerical and label columns

In [122]:
numerical = ['Date','Hour','WindSpeed','WindDirection']
categorical = ['SiteName','SiteType','Month','DayOfWeek']
input_features = numerical + categorical
label = 'ParticulateReading'

#Create train and test data

As the distribution in the dataset is not even across all site types, we will startify the sampling by site type.

In [123]:
#create sub-lists on Urban Background, Rural Background and Urban Traffic

df_UB=df_aq.filter(df_aq.SiteType=='Urban Background')
df_RB=df_aq.filter(df_aq.SiteType=='Rural Background')
df_UT=df_aq.filter(df_aq.SiteType=='Urban Traffic')

#Split into train and test using reandom split
ubTrain,ubTest = df_UB.randomSplit(weights=[0.85,0.15],seed=42)
rbTrain,rbTest = df_RB.randomSplit(weights=[0.85,0.15],seed=42)
utTrain,utTest = df_UT.randomSplit(weights=[0.85,0.15],seed=42)

#recombine into overall test and train
df_train=ubTrain.union(rbTrain).union(utTrain)
df_test=ubTest.union(rbTest).union(utTest)

#count rows to check split
print(df_aq.count())
print(df_train.count())
print(df_test.count())

96360
82078
14282


# New Section

# Create pipeline to prepare data and run selected model on it

#Run pipeline on Test data